# Chapter 14: Claims Processing Pipeline

**From Enterprise Architect to AI Enterprise Architect**

Build a simplified insurance claims processing pipeline that demonstrates
multi-stage AI orchestration: ingest a claim document, extract key entities,
classify urgency, estimate damage, detect fraud signals, and generate a
settlement recommendation with explanation.

**Key Insight:** Multi-model pipelines with confidence-based automation
balance speed and safety. Not every claim needs a human — but the system
must know when one does.

---

## Setup

This notebook uses the OpenAI API. You'll need an API key.

In [ ]:
# Install dependencies
!pip install -q openai tabulate

In [ ]:
import os
import json
from typing import Dict, List, Optional
from dataclasses import dataclass
from tabulate import tabulate

# ============================================================
# API Key Configuration
# ============================================================
# Option 1: Set your key as an environment variable before running:
#   export OPENAI_API_KEY="sk-..."
#
# Option 2 (Colab): Use Colab secrets or set it directly:
#   import os
#   os.environ["OPENAI_API_KEY"] = "sk-..."  # Don't commit this!
# ============================================================

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "Please set the OPENAI_API_KEY environment variable. "
        "In Colab, use: os.environ['OPENAI_API_KEY'] = 'sk-...'"
    )

from openai import OpenAI
client = OpenAI(api_key=api_key)

# Model to use throughout the notebook
MODEL = "gpt-4o-mini"
print(f"Using model: {MODEL}")

## Sample Claims

We define five insurance claims of varying complexity. In production, these
would come from intake forms, emails, scanned documents, or phone transcripts.

In [ ]:
SAMPLE_CLAIMS = [
    {
        "id": "CLM-001",
        "label": "Simple fender bender",
        "text": (
            "Claim submitted by John Smith on 2025-11-15. "
            "While stopped at a red light on Main Street, another vehicle "
            "rear-ended my 2022 Honda Civic. Minor damage to the rear bumper "
            "and tail light. No injuries. Police report #45-2025-1102 filed. "
            "Estimated repair cost: $1,200. Photos attached showing bumper "
            "damage and cracked tail light assembly."
        ),
    },
    {
        "id": "CLM-002",
        "label": "Moderate property damage",
        "text": (
            "Claim submitted by Maria Garcia on 2025-11-20. "
            "A severe windstorm on November 18 caused a large oak tree to "
            "fall onto my garage at 456 Elm Avenue, crushing the roof and "
            "damaging my 2021 Toyota RAV4 parked inside. The garage roof "
            "needs complete replacement and the car has significant roof and "
            "hood damage. Contractor estimate for garage: $18,500. Auto body "
            "shop estimate for car: $9,200. Total claimed: $27,700. "
            "Photos and contractor quotes attached."
        ),
    },
    {
        "id": "CLM-003",
        "label": "Complex multi-vehicle accident",
        "text": (
            "Claim submitted by David Chen on 2025-12-01. "
            "Multi-vehicle accident on Interstate 95 during heavy rain. "
            "My 2023 BMW X5 was in the middle lane when a semi-truck "
            "jackknifed ahead causing a chain reaction. My vehicle was struck "
            "by a pickup truck from behind and pushed into the sedan ahead. "
            "Significant front and rear damage to my vehicle — likely totaled. "
            "I was taken to Memorial Hospital with whiplash and a fractured "
            "wrist. Medical bills so far: $12,400. Vehicle value: $52,000. "
            "Lost wages (2 weeks): $4,800. Three other vehicles involved. "
            "Police report and hospital records attached. Total claimed: $69,200."
        ),
    },
    {
        "id": "CLM-004",
        "label": "Medical claim",
        "text": (
            "Claim submitted by Sarah Johnson on 2025-12-05. "
            "I slipped on an unmarked wet floor at Riverside Mall on "
            "December 2 and fell, injuring my back and left knee. I was "
            "transported by ambulance to City General Hospital where X-rays "
            "and an MRI were performed. Diagnosed with a herniated disc "
            "(L4-L5) and torn meniscus. Orthopedic surgeon recommends "
            "arthroscopic surgery for the knee. Current medical bills: $8,700. "
            "Estimated surgery and rehab: $25,000. Lost wages for projected "
            "8-week recovery: $9,600. Total claimed: $43,300. "
            "Medical records and incident report from mall security attached."
        ),
    },
    {
        "id": "CLM-005",
        "label": "Suspicious / potential fraud",
        "text": (
            "Claim submitted by Robert Williams on 2025-12-10. "
            "My 2024 Mercedes-Benz S-Class was stolen from my driveway "
            "on December 8. I noticed it missing at 7 AM. No security "
            "camera footage available — cameras were not working that week "
            "due to a power outage. Police report filed (#PR-2025-4421). "
            "The vehicle was purchased 3 months ago for $95,000. I recently "
            "increased my coverage limit last month. There are two outstanding "
            "loans on the vehicle totaling $82,000. I also need coverage for "
            "$15,000 worth of personal electronics that were in the car. "
            "Total claimed: $110,000."
        ),
    },
]

print(f"Loaded {len(SAMPLE_CLAIMS)} sample claims.")
for claim in SAMPLE_CLAIMS:
    print(f"  {claim['id']}: {claim['label']}")

## Helper: Call OpenAI with JSON Response

We use a helper function that sends a system + user prompt to the model and
parses the result as JSON. Each pipeline stage uses this pattern.

In [ ]:
def call_llm(system_prompt: str, user_prompt: str) -> dict:
    """Call the OpenAI API and parse the response as JSON."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
        temperature=0.1,  # Low temperature for consistency
    )
    return json.loads(response.choices[0].message.content)

## The Claims Pipeline

The `ClaimsPipeline` class implements five stages, each backed by an LLM call
with a tailored prompt:

1. **Entity Extraction** — Pull structured data from unstructured claim text
2. **Urgency Classification** — Low / Medium / High with a confidence score
3. **Damage Estimation** — Independent estimate with reasoning
4. **Fraud Detection** — Score 0-1 with specific red flags
5. **Settlement Generation** — Recommendation with plain-language explanation

After all stages, a **routing decision** determines whether the claim can be
auto-approved or must be sent to a human reviewer.

In [ ]:
class ClaimsPipeline:
    """Multi-stage AI claims processing pipeline."""

    def extract_entities(self, claim_text: str) -> dict:
        """Stage 1: Extract structured entities from claim text."""
        system = (
            "You are an insurance claim entity extractor. Extract the following "
            "fields from the claim text and return valid JSON:\n"
            "- claimant_name (string)\n"
            "- incident_date (string, YYYY-MM-DD)\n"
            "- damage_type (string: 'auto', 'property', 'medical', 'theft', or 'multi')\n"
            "- claimed_amount (number, total dollars claimed)\n"
            "- description_summary (string, one-sentence summary)\n"
            "If a field cannot be determined, use null."
        )
        return call_llm(system, claim_text)

    def classify_urgency(self, claim_text: str) -> dict:
        """Stage 2: Classify urgency as low/medium/high with confidence."""
        system = (
            "You are an insurance claim urgency classifier. Analyze the claim "
            "and return JSON with:\n"
            "- urgency (string: 'low', 'medium', or 'high')\n"
            "- confidence (float 0.0 to 1.0)\n"
            "- reasoning (string, brief explanation)\n\n"
            "Guidelines:\n"
            "- low: minor damage, no injuries, clear liability, < $5,000\n"
            "- medium: moderate damage, minor injuries, $5,000-$50,000\n"
            "- high: severe injuries, complex liability, > $50,000, or fraud indicators"
        )
        return call_llm(system, claim_text)

    def estimate_damage(self, claim_text: str, entities: dict) -> dict:
        """Stage 3: Independently estimate the damage amount."""
        context = f"Extracted entities: {json.dumps(entities)}"
        system = (
            "You are an insurance damage estimator. Based on the claim text "
            "and extracted entities, provide an independent damage estimate. "
            "Return JSON with:\n"
            "- estimated_amount (number, your best estimate in dollars)\n"
            "- confidence (float 0.0 to 1.0)\n"
            "- reasoning (string, explain how you arrived at the estimate)\n"
            "- variance_from_claimed (string, e.g. '+10%' or '-5%' or 'aligned')"
        )
        return call_llm(system, f"{claim_text}\n\n{context}")

    def detect_fraud_signals(self, claim_text: str, entities: dict) -> dict:
        """Stage 4: Detect potential fraud signals."""
        context = f"Extracted entities: {json.dumps(entities)}"
        system = (
            "You are an insurance fraud detection analyst. Analyze the claim "
            "for fraud indicators. Return JSON with:\n"
            "- fraud_score (float 0.0 to 1.0, where 0 = no suspicion, 1 = highly suspicious)\n"
            "- flags (list of strings, each a specific red flag found)\n"
            "- recommendation (string: 'clear', 'monitor', or 'investigate')\n\n"
            "Common red flags: recent policy changes, disabled security, "
            "inconsistent details, excessive claimed amounts, financial pressure indicators."
        )
        return call_llm(system, f"{claim_text}\n\n{context}")

    def generate_settlement(self, entities: dict, urgency: dict,
                            estimate: dict, fraud: dict) -> dict:
        """Stage 5: Generate a settlement recommendation."""
        context = json.dumps({
            "entities": entities,
            "urgency": urgency,
            "damage_estimate": estimate,
            "fraud_analysis": fraud,
        }, indent=2)
        system = (
            "You are an insurance settlement advisor. Based on the pipeline "
            "analysis results, generate a settlement recommendation. "
            "Return JSON with:\n"
            "- recommended_action (string: 'approve', 'partial_approve', "
            "'negotiate', 'deny', or 'escalate')\n"
            "- settlement_amount (number or null)\n"
            "- explanation (string, plain-language explanation suitable for "
            "the claims adjuster)\n"
            "- conditions (list of strings, any conditions on the settlement)"
        )
        return call_llm(system, context)

    def route_claim(self, entities: dict, urgency: dict, fraud: dict) -> dict:
        """Determine routing: auto-approve or send to human reviewer."""
        claimed = entities.get("claimed_amount", float("inf")) or float("inf")
        confidence = urgency.get("confidence", 0)
        fraud_score = fraud.get("fraud_score", 1)

        auto_approve = (
            confidence > 0.95
            and claimed < 2000
            and fraud_score < 0.2
        )

        return {
            "routing": "AUTO-APPROVE" if auto_approve else "HUMAN REVIEW",
            "reason": (
                "Low amount, high confidence, no fraud signals"
                if auto_approve
                else self._routing_reason(claimed, confidence, fraud_score)
            ),
        }

    @staticmethod
    def _routing_reason(amount, confidence, fraud_score) -> str:
        reasons = []
        if amount >= 2000:
            reasons.append(f"amount ${amount:,.0f} >= $2,000 threshold")
        if confidence <= 0.95:
            reasons.append(f"confidence {confidence:.2f} <= 0.95")
        if fraud_score >= 0.2:
            reasons.append(f"fraud score {fraud_score:.2f} >= 0.20")
        return "; ".join(reasons) if reasons else "multiple factors"

    def process_claim(self, claim: dict) -> dict:
        """Run a single claim through the full pipeline."""
        claim_text = claim["text"]
        claim_id = claim["id"]

        print(f"\n{'='*60}")
        print(f"  Processing {claim_id}: {claim['label']}")
        print(f"{'='*60}")

        # Stage 1
        print("  [1/5] Extracting entities...")
        entities = self.extract_entities(claim_text)

        # Stage 2
        print("  [2/5] Classifying urgency...")
        urgency = self.classify_urgency(claim_text)

        # Stage 3
        print("  [3/5] Estimating damage...")
        estimate = self.estimate_damage(claim_text, entities)

        # Stage 4
        print("  [4/5] Detecting fraud signals...")
        fraud = self.detect_fraud_signals(claim_text, entities)

        # Stage 5
        print("  [5/5] Generating settlement recommendation...")
        settlement = self.generate_settlement(entities, urgency, estimate, fraud)

        # Routing
        routing = self.route_claim(entities, urgency, fraud)

        return {
            "claim_id": claim_id,
            "label": claim["label"],
            "entities": entities,
            "urgency": urgency,
            "estimate": estimate,
            "fraud": fraud,
            "settlement": settlement,
            "routing": routing,
        }

## Run the Pipeline

Process all five sample claims through the pipeline. Each claim goes through
all five stages sequentially — in production, independent stages could run
in parallel for throughput.

In [ ]:
pipeline = ClaimsPipeline()

# Process all claims
results = []
for claim in SAMPLE_CLAIMS:
    result = pipeline.process_claim(claim)
    results.append(result)

print(f"\nProcessed {len(results)} claims.")

## Display Results: Entity Extraction

See what structured data the model extracted from each claim's free-text description.

In [ ]:
# Entity extraction results
print("ENTITY EXTRACTION RESULTS")
print("=" * 70)
headers = ["Claim", "Claimant", "Date", "Type", "Amount"]
rows = []
for r in results:
    e = r["entities"]
    rows.append([
        r["claim_id"],
        e.get("claimant_name", "N/A"),
        e.get("incident_date", "N/A"),
        e.get("damage_type", "N/A"),
        f"${e.get('claimed_amount', 0):,.0f}" if e.get("claimed_amount") else "N/A",
    ])
print(tabulate(rows, headers=headers, tablefmt="grid"))

## Display Results: Urgency and Fraud Analysis

In [ ]:
# Urgency classification
print("URGENCY CLASSIFICATION")
print("=" * 70)
headers = ["Claim", "Label", "Urgency", "Confidence", "Reasoning"]
rows = []
for r in results:
    u = r["urgency"]
    rows.append([
        r["claim_id"],
        r["label"][:25],
        u.get("urgency", "N/A").upper(),
        f"{u.get('confidence', 0):.2f}",
        u.get("reasoning", "N/A")[:60],
    ])
print(tabulate(rows, headers=headers, tablefmt="grid"))

print()

# Fraud detection
print("FRAUD ANALYSIS")
print("=" * 70)
headers = ["Claim", "Fraud Score", "Recommendation", "Flags"]
rows = []
for r in results:
    f = r["fraud"]
    flags = f.get("flags", [])
    flag_text = "; ".join(flags[:3]) if flags else "None"
    rows.append([
        r["claim_id"],
        f"{f.get('fraud_score', 0):.2f}",
        f.get("recommendation", "N/A").upper(),
        flag_text[:60],
    ])
print(tabulate(rows, headers=headers, tablefmt="grid"))

## Display Results: Damage Estimates and Settlements

In [ ]:
# Damage estimates
print("DAMAGE ESTIMATES")
print("=" * 70)
headers = ["Claim", "Claimed", "Estimated", "Variance", "Confidence"]
rows = []
for r in results:
    e = r["entities"]
    est = r["estimate"]
    rows.append([
        r["claim_id"],
        f"${e.get('claimed_amount', 0):,.0f}" if e.get("claimed_amount") else "N/A",
        f"${est.get('estimated_amount', 0):,.0f}",
        est.get("variance_from_claimed", "N/A"),
        f"{est.get('confidence', 0):.2f}",
    ])
print(tabulate(rows, headers=headers, tablefmt="grid"))

print()

# Settlement recommendations
print("SETTLEMENT RECOMMENDATIONS")
print("=" * 70)
for r in results:
    s = r["settlement"]
    print(f"\n{r['claim_id']} ({r['label']}):")
    print(f"  Action: {s.get('recommended_action', 'N/A').upper()}")
    amt = s.get('settlement_amount')
    if amt:
        print(f"  Amount: ${amt:,.0f}")
    print(f"  Explanation: {s.get('explanation', 'N/A')}")
    conditions = s.get("conditions", [])
    if conditions:
        print(f"  Conditions:")
        for c in conditions:
            print(f"    - {c}")

## Routing Decisions

The routing logic applies confidence-based rules:

- **Auto-approve** if: confidence > 0.95 AND amount < $2,000 AND fraud score < 0.2
- **Human review** otherwise

This is the critical architectural decision: the system knows what it doesn't know.

In [ ]:
print("ROUTING DECISIONS")
print("=" * 70)
headers = ["Claim", "Label", "Amount", "Confidence", "Fraud", "Routing", "Reason"]
rows = []
for r in results:
    e = r["entities"]
    u = r["urgency"]
    f = r["fraud"]
    rt = r["routing"]
    rows.append([
        r["claim_id"],
        r["label"][:20],
        f"${e.get('claimed_amount', 0):,.0f}" if e.get("claimed_amount") else "N/A",
        f"{u.get('confidence', 0):.2f}",
        f"{f.get('fraud_score', 0):.2f}",
        rt["routing"],
        rt["reason"][:40],
    ])
print(tabulate(rows, headers=headers, tablefmt="grid"))

# Summary
auto_count = sum(1 for r in results if r["routing"]["routing"] == "AUTO-APPROVE")
human_count = len(results) - auto_count
print(f"\nSummary: {auto_count} auto-approved, {human_count} routed to human review")
print(f"Automation rate: {auto_count/len(results)*100:.0f}%")

## Pipeline Summary

Let's generate a complete summary view of all claims processed.

In [ ]:
print("\n" + "=" * 70)
print("  COMPLETE PIPELINE SUMMARY")
print("=" * 70)

for r in results:
    e = r["entities"]
    u = r["urgency"]
    est = r["estimate"]
    f = r["fraud"]
    s = r["settlement"]
    rt = r["routing"]

    print(f"\n--- {r['claim_id']}: {r['label']} ---")
    print(f"  Claimant: {e.get('claimant_name', 'N/A')}")
    print(f"  Claimed: ${e.get('claimed_amount', 0):,.0f}  |  "
          f"Estimated: ${est.get('estimated_amount', 0):,.0f}")
    print(f"  Urgency: {u.get('urgency', 'N/A').upper()} "
          f"(conf: {u.get('confidence', 0):.2f})")
    print(f"  Fraud Score: {f.get('fraud_score', 0):.2f} "
          f"({f.get('recommendation', 'N/A')})")
    print(f"  Settlement: {s.get('recommended_action', 'N/A').upper()}")
    amt = s.get('settlement_amount')
    if amt:
        print(f"  Settlement Amount: ${amt:,.0f}")
    print(f"  Routing: ** {rt['routing']} **")
    print(f"  Reason: {rt['reason']}")

## Key Takeaways

1. **Pipeline architecture** breaks complex decisions into auditable stages.
   Each stage has a clear input, output, and responsibility.

2. **Confidence-based routing** is the linchpin of production AI. The system
   must know when it's uncertain and defer to humans.

3. **Fraud detection as a separate stage** means it can be tuned independently
   without affecting entity extraction or damage estimation.

4. **Structured JSON outputs** from each stage make the pipeline composable
   and testable — you can unit test each stage independently.

5. **The auto-approve threshold is a business decision**, not a technical one.
   The architect's job is to make those thresholds visible and adjustable.

In production, you would add: logging, retry logic, cost tracking per API call,
A/B testing of prompts, and a feedback loop from human reviewers back to the model.